# Full anonymization pipeline demo

Separation -> blurring -> remixing using `source_separation` + `voice_blurring`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import librosa
import numpy as np
from IPython.display import Audio, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "anonymization_pipeline").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "anonymization_pipeline").is_dir():
    raise RuntimeError("Run this notebook from the repo root or from repo_root/notebooks")

sys.path.insert(0, str(REPO_ROOT))

from anonymization_pipeline import anonymize_audio
from source_separation import load_unet_checkpoint


In [3]:
AUDIO_PATH = REPO_ROOT / "notebooks" / "mix_demo.wav"
CKPT_PATH = REPO_ROOT / "checkpoints/unet_run1/unet_voice_sep.pt"
BLUR_MODES = ["low_pass", "mfcc"]

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"Missing input audio: {AUDIO_PATH}")
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

y, sr = librosa.load(str(AUDIO_PATH), sr=None, mono=True)
y = np.asarray(y, dtype=np.float32)
model, config, device = load_unet_checkpoint(CKPT_PATH, device="auto")

results = {}
for mode in BLUR_MODES:
    res = anonymize_audio(y, sr, model=model, config=config, device=device, blur_mode=mode)
    results[mode] = res
    print(f"Device: {device} | blur_mode={res.blur_mode} | sr={res.sr}")
    print(
        f"lens: mix={len(y)} voice={len(res.voice_est)} "
        f"blurred={len(res.blurred_voice)} remix={len(res.anonymized_mix)}"
    )


Device: cuda | blur_mode=low_pass | sr=16000
lens: mix=160000 voice=160000 blurred=160000 remix=160000
Device: cuda | blur_mode=mfcc | sr=16000
lens: mix=160000 voice=160000 blurred=160000 remix=160000


In [4]:
print("Input mix")
display(Audio(y, rate=sr))

for mode in BLUR_MODES:
    res = results[mode]
    print(f"\n=== {mode} ===")
    print("Separated voice")
    display(Audio(res.voice_est, rate=res.sr))
    print("Blurred voice")
    display(Audio(res.blurred_voice, rate=res.sr))
    print("Anonymized remix")
    display(Audio(res.anonymized_mix, rate=res.sr))


Input mix



=== low_pass ===
Separated voice


Blurred voice


Anonymized remix



=== mfcc ===
Separated voice


Blurred voice


Anonymized remix
